In [0]:
%run ../utils/adls_auth

In [0]:
%run ../utils/control_table

In [0]:
%run ../utils/dq_helpers

In [0]:
spark.conf.set("spark.databricks.delta.properties.defaults.enableDeletionVectors", "false")

In [0]:
import uuid
from datetime import datetime
from pyspark.sql.functions import col, to_timestamp, current_timestamp, lit, array, arrays_overlap
from delta.tables import DeltaTable

BRONZE_PATH = "abfss://bronze@stdatalakenyctaxi.dfs.core.windows.net/trips_stream_raw"
CHECKPOINT_PATH = "abfss://silver@stdatalakenyctaxi.dfs.core.windows.net/_checkpoints/trips_stream_silver"
SILVER_PATH = "abfss://silver@stdatalakenyctaxi.dfs.core.windows.net/trips_stream_silver"
QUARANTINE_PATH = "abfss://silver@stdatalakenyctaxi.dfs.core.windows.net/trips_stream_quarantine"

silver_table_exists = DeltaTable.isDeltaTable(spark, SILVER_PATH)



In [0]:

def process_batch(batch_df, batch_id):
    global silver_table_exists
    run_batch_id = f"{batch_id}_{uuid.uuid4()}"

    if batch_df.count() == 0:
        print(f"[batch {batch_id}] empty micro-batch, skipping.")
        return

    checks = {
        "null_pickup_location": col("pickup_location_id").isNull(),
        "negative_fare": col("fare_amount") < 0,
        "negative_distance": col("trip_distance") < 0,
    }
    flagged_df = flag_row_level_checks(batch_df, checks)
    critical_array = array(*[lit(c) for c in checks.keys()])
    flagged_df = flagged_df.withColumn("_has_critical_failure", arrays_overlap(col("_dq_failures"), critical_array))

    quarantine_df = flagged_df.filter(col("_has_critical_failure")).withColumn("_quarantined_at", current_timestamp())
    clean_df = flagged_df.filter(~col("_has_critical_failure"))

    critical_failed = quarantine_df.count()
    log_dq_result(
        spark, run_batch_id, "silver", "trips_stream_silver", "critical_row_checks",
        severity="CRITICAL", rows_checked=batch_df.count(), rows_failed=critical_failed,
        action_taken="Rows with any critical failure routed to trips_stream_quarantine.",
    )

    if not silver_table_exists:
        clean_df.write.format("delta").mode("overwrite").save(SILVER_PATH)
        silver_table_exists = True
    else:
        silver_table = DeltaTable.forPath(spark, SILVER_PATH)
        merge_cond = "target.vendor_id = source.vendor_id AND target.pickup_location_id = source.pickup_location_id AND target.event_time = source.event_time"
        (silver_table.alias("target")
            .merge(clean_df.alias("source"), merge_cond)
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute())

    quarantine_df.write.format("delta").mode("append").save(QUARANTINE_PATH)
    log_ingestion_event(
        spark, run_batch_id, "trips_stream_silver_transform", f"batch_{batch_id}",
        status="SUCCESS", rows_written=clean_df.count(), started_at=datetime.utcnow(),
    )
    print(f"[batch {batch_id}] merged {clean_df.count()} rows, quarantined {critical_failed}.")


In [0]:


bronze_stream = (
    spark.readStream.format("delta").load(BRONZE_PATH)
    .withColumn("event_timestamp", to_timestamp(col("event_time")))
    .withWatermark("event_timestamp", "10 minutes") 
    .dropDuplicates(["vendor_id", "pickup_location_id", "dropoff_location_id", "event_time"])
)

query = (
    bronze_stream.writeStream
    .foreachBatch(process_batch)
    .option("checkpointLocation", CHECKPOINT_PATH)
    .trigger(processingTime="2 minutes")  
    .start()
)
print(f"Streaming Silver query started: {query.id}")
print("Remember: query.stop() when done testing — this runs continuously.")

In [0]:
# query.stop()